# 🏥 Retrain & Fine-Tune Existing `best.pt` Model
### Google Colab (T4 GPU)

> **Note:** This notebook loads and retrains your **existing `best.pt` model** from `models/`. It does **not** create or download a new base model from scratch.

**GPU Setup:** Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Verify T4 GPU is assigned
!nvidia-smi

In [ ]:
# 2. Install Ultralytics YOLO
!pip install -q ultralytics

In [ ]:
# 3. Upload dataset.zip
# Note: dataset.zip already includes your updated labels and existing best.pt
from google.colab import files
import os

if not os.path.exists('/content/dataset.zip'):
    print('Please upload dataset.zip:')
    uploaded = files.upload()
else:
    print('dataset.zip is already present in /content/')

In [ ]:
# 4. Unzip dataset and extract existing best.pt
!unzip -q -o /content/dataset.zip -d /content/

# Ensure best.pt is at /content/best.pt
if os.path.exists('/content/dataset/best.pt') and not os.path.exists('/content/best.pt'):
    !cp /content/dataset/best.pt /content/best.pt

# If best.pt was not in zip, allow uploading it directly:
if not os.path.exists('/content/best.pt'):
    print('Upload your existing best.pt from backend/models/best.pt:')
    files.upload()

print('✓ dataset and existing best.pt ready!')
!cat /content/dataset/data.yaml

In [ ]:
# 5. Load EXISTING best.pt model (NO new model created)
from ultralytics import YOLO

# Load your existing model directly
model = YOLO('/content/best.pt')

print('✓ Loaded existing model successfully!')
print('Existing classes:', model.names)

In [ ]:
# 6. Fine-Tune Existing best.pt on the new dataset using T4 GPU
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=1280,
    batch=8,
    device=0,             # T4 GPU
    optimizer='AdamW',
    lr0=0.001,            # Tuned learning rate for fine-tuning pre-trained weights
    name='finetuned_medanta_model',
    save=True,
    plots=True
)

In [ ]:
# 7. Download the newly fine-tuned best.pt
from google.colab import files
import os

weights_path = 'runs/detect/finetuned_medanta_model/weights/best.pt'
if os.path.exists(weights_path):
    print(f'✓ Training complete! Downloading {weights_path}...')
    files.download(weights_path)
    print('(If automatic download is blocked by your browser, open the left Files panel, navigate to ' + weights_path + ', right-click and select Download)')
else:
    print(f'Warning: {weights_path} not found. Check the runs/detect/ folder.')